# Задание 6. Итоговая рекомендательная система — гибрид нескольких подходов

## Постановка задачи

Необходимо разработать итоговую рекомендательную систему для Goodbooks-10k Extended, объединив несколько подходов, рассмотренных в предыдущих заданиях.

Обязательным условием является использование Content-Based.

В работе исследуются две последовательные схемы:

1. **SVD → User+Item KNN → Content-Based**
2. **Content-Based → User+Item KNN**

В первом варианте SVD быстро формирует набор кандидатов, User+Item KNN уточняет его по пользовательским взаимодействиям, а Content-Based выполняет финальную персонализацию на основе содержания книг.

Во втором варианте Content-Based сначала ограничивает пространство содержательно подходящими книгами, после чего User+Item KNN выбирает наиболее подходящие варианты с учётом поведения пользователей.

Таким образом, модели не складывают свои оценки между собой. Каждый этап последовательно решает отдельную задачу.


## Структура и анализ данных

Используется Goodbooks-10k Extended.

Файл `ratings.csv` содержит пользовательские оценки: `user_id`, `book_id`, `rating`.

Файл `books_enriched.csv` содержит признаки книг. Для Content-Based используются `title`, `authors`, `genres` и `description`.

Для всех методов используется одинаковое случайное разбиение 80 процентов на обучение и 20 процентов на тестирование с `random_state = 42`.

Релевантной считается книга с тестовой оценкой не ниже 4.

Основные метрики:

* Precision@K;
* Recall@K;
* NDCG@K;
* HitRate@K.

Используются значения K = 5, 10 и 20. Основным критерием сравнения является NDCG@10.


In [1]:
import time, math, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import linear_kernel
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import NearestNeighbors
warnings.filterwarnings('ignore')



In [22]:
DATA_DIR = 'goodbooks-10k-extended'
PARAMS = {
    'TEST_SIZE':0.20,'RANDOM_STATE':42,'TOP_K_VALUES':[5,10,20],
    'RELEVANCE_THRESHOLD':4,'N_EVAL_USERS':200,
    'SVD_COMPONENTS':10,'SVD_CANDIDATES':[300,500,800],
    'USER_K':[20,50],'ITEM_K':[20,50],
    'KNN_POOL':[50,100,200],
    'CB_MIN_LIKE':3,'CB_PROFILE_ITEMS':50,
    'CB_NGRAM_RANGE':(1,5),'CB_MAX_FEATURES':30000,
    'CB_CANDIDATES':[200,500,800,1000,1250]
}
np.random.seed(PARAMS['RANDOM_STATE'])

## Загрузка и анализ данных

Сначала загружаются оценки и метаданные книг.

Проверяются число пользователей, число книг, количество оценок, разреженность матрицы взаимодействий и наличие текстовых признаков.


In [2]:
def load_data():
    ratings = pd.read_csv(f'{DATA_DIR}/ratings.csv',usecols=['user_id','book_id','rating'])
    books = pd.read_csv(f'{DATA_DIR}/books_enriched.csv')
    if 'book_id' not in books.columns: books = books.rename(columns={books.columns[0]:'book_id'})
    ratings = ratings.dropna(subset=['user_id','book_id','rating']).copy()
    books = books.dropna(subset=['book_id']).copy()
    ratings[['user_id','book_id']] = ratings[['user_id','book_id']].astype(np.int32)
    ratings['rating'] = ratings['rating'].astype(np.float32)
    books['book_id'] = books['book_id'].astype(np.int32)
    for col in ['title','authors','genres','description']:
        if col not in books.columns: books[col] = ''
        books[col] = books[col].fillna('').astype(str)
    return ratings,books

ratings_df,books_df = load_data()
n_users,n_books,n_ratings = ratings_df['user_id'].nunique(),ratings_df['book_id'].nunique(),len(ratings_df)
sparsity = (1-n_ratings/(n_users*n_books))*100

print('='*60)
print('АНАЛИЗ GOODBOOKS-10K EXTENDED')
print('='*60)
print(f'Уникальных пользователей: {n_users:,}')
print(f'Уникальных книг:          {n_books:,}')
print(f'Всего оценок:             {n_ratings:,}')
print(f'Разреженность матрицы:    {sparsity:.2f}%')
print(f'Средняя оценка:           {ratings_df["rating"].mean():.3f}')
print('='*60)
display(books_df[['title','authors','genres','description']].head())

АНАЛИЗ GOODBOOKS-10K EXTENDED
Уникальных пользователей: 53,424
Уникальных книг:          10,000
Всего оценок:             5,976,479
Разреженность матрицы:    98.88%
Средняя оценка:           3.920


,title,authors,genres,description
0,"The Hunger Games (The Hunger Games, #1)",['Suzanne Collins'],"['young-adult', 'fiction', 'fantasy', 'science...",WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...
1,Harry Potter and the Sorcerer's Stone (Harry P...,"['J.K. Rowling', 'Mary GrandPré']","['fantasy', 'fiction', 'young-adult', 'classics']",Harry Potter's life is miserable. His parents ...
2,"Twilight (Twilight, #1)",['Stephenie Meyer'],"['young-adult', 'fantasy', 'romance', 'fiction...",About three things I was absolutely positive.\...
3,To Kill a Mockingbird,['Harper Lee'],"['classics', 'fiction', 'historical-fiction', ...",The unforgettable novel of a childhood in a sl...
4,The Great Gatsby,['F. Scott Fitzgerald'],"['classics', 'fiction', 'historical-fiction', ...",Alternate Cover Edition ISBN: 0743273567 (ISBN...


## Подготовка обучающей и тестовой выборок

Используется то же разбиение, что и в предыдущих работах: 80 процентов данных используются для обучения и 20 процентов для тестирования.

Все компоненты гибридной модели строятся только по обучающей выборке.

Для ускорения экспериментов итоговое качество оценивается на случайной выборке из 200 пользователей, имеющих хотя бы одну релевантную книгу в тестовой выборке.


In [3]:
train_df,test_df = train_test_split(ratings_df,test_size=PARAMS['TEST_SIZE'],random_state=PARAMS['RANDOM_STATE'])

book_ids = pd.Index(np.sort(train_df['book_id'].unique()))
book_to_idx = pd.Series(np.arange(len(book_ids)),index=book_ids)
user_ids = pd.Index(np.sort(train_df['user_id'].unique()))
user_to_idx = pd.Series(np.arange(len(user_ids)),index=user_ids)

rows = train_df['user_id'].map(user_to_idx).to_numpy()
cols = train_df['book_id'].map(book_to_idx).to_numpy()
data = train_df['rating'].to_numpy(dtype=np.float32)
R = csr_matrix((data,(rows,cols)),shape=(len(user_ids),len(book_ids)))

test_relevant = {uid:set(g.loc[g['rating']>=PARAMS['RELEVANCE_THRESHOLD'],'book_id']) for uid,g in test_df.groupby('user_id')}
candidate_users = np.array([uid for uid in train_df['user_id'].unique() if uid in test_relevant and len(test_relevant[uid])>0])
sample_eval_users = np.random.choice(candidate_users,size=min(PARAMS['N_EVAL_USERS'],len(candidate_users)),replace=False)

user_row_cache = {int(uid):int(user_to_idx[uid]) for uid in sample_eval_users}
seen_cache = {int(uid):set(train_df.loc[train_df['user_id']==uid,'book_id']) for uid in sample_eval_users}

print(f'Обучающая выборка: {len(train_df):,}')
print(f'Тестовая выборка:   {len(test_df):,}')
print(f'Книг в обучении:    {len(book_ids):,}')
print(f'Пользователей для оценки: {len(sample_eval_users)}')

Обучающая выборка: 4,781,183
Тестовая выборка:   1,195,296
Книг в обучении:    10,000
Пользователей для оценки: 200


## Этап 1. SVD — генерация кандидатов

SVD используется как быстрый первый этап.

Для каждого пользователя строится латентное представление, после чего книги ранжируются по восстановленной оценке.

На этом этапе не формируется окончательная рекомендация. SVD только сокращает исходное пространство книг до нескольких сотен кандидатов, которые передаются следующим этапам.


In [4]:
svd_model = TruncatedSVD(n_components=PARAMS['SVD_COMPONENTS'],n_iter=7,random_state=PARAMS['RANDOM_STATE'])
svd_start = time.time()
USER_FACTORS = svd_model.fit_transform(R)
print(f'SVD построен за {time.time()-svd_start:.1f} сек.; компонент: {PARAMS["SVD_COMPONENTS"]}; объяснённая дисперсия: {svd_model.explained_variance_ratio_.sum():.4f}')

def get_svd_candidates(uid,n_candidates=500):
    uidx = user_row_cache[int(uid)]
    scores = USER_FACTORS[uidx] @ svd_model.components_
    seen = seen_cache[int(uid)]
    need = min(n_candidates+len(seen),len(book_ids))
    order = np.argsort(-scores)[:need]
    return [int(book_ids[i]) for i in order if int(book_ids[i]) not in seen][:n_candidates]

SVD построен за 0.4 сек.; компонент: 10; объяснённая дисперсия: 0.0899


## Этап 2. User+Item KNN

KNN используется как отдельный этап уточнения кандидатов.

User-Based часть использует похожих пользователей, а Item-Based часть — похожесть книг по структуре пользовательских оценок.

Для кандидатов рассчитываются два независимых сигнала. Они используются внутри самого KNN-этапа для формирования итогового KNN-порядка.

Важно, что KNN не складывает свою оценку с SVD или Content-Based. Он работает только с кандидатами, полученными на предыдущем этапе.


In [5]:
USER_K = 50
ITEM_K = 50

user_knn = NearestNeighbors(n_neighbors=USER_K+1,metric='cosine',algorithm='brute',n_jobs=-1).fit(R)
item_knn = NearestNeighbors(n_neighbors=ITEM_K+1,metric='cosine',algorithm='brute',n_jobs=-1).fit(R.T)

print('User-Based и Item-Based KNN готовы.')

User-Based и Item-Based KNN готовы.


In [13]:
MAX_USER_K = max(PARAMS['USER_K'])
MAX_ITEM_K = max(PARAMS['ITEM_K'])
USER_NEIGHBOR_CACHE = {}
ITEM_NEIGHBOR_CACHE = {}

print('Кэширование User-KNN...')
start = time.time()

eval_rows = np.array([user_row_cache[int(uid)] for uid in sample_eval_users])
distances, neighbors = user_knn.kneighbors(R[eval_rows], n_neighbors=MAX_USER_K+1)

for i,uid in enumerate(sample_eval_users):
    USER_NEIGHBOR_CACHE[int(uid)] = (neighbors[i][1:], np.maximum(1-distances[i][1:],0))

print(f'User-KNN готов: {len(USER_NEIGHBOR_CACHE)} пользователей за {time.time()-start:.1f} сек.')

liked_ids = set()
for uid in sample_eval_users:
    liked = train_df[(train_df['user_id']==uid)&(train_df['rating']>=4)].sort_values('rating',ascending=False).head(PARAMS['CB_PROFILE_ITEMS'])
    liked_ids.update(int(x) for x in liked['book_id'] if x in book_to_idx.index)

liked_ids = list(liked_ids)
print(f'\nУникальных книг для Item-KNN: {len(liked_ids):,}')
print('Кэширование Item-KNN...')

start = time.time()
chunk_size = 100

for start_idx in range(0,len(liked_ids),chunk_size):
    chunk = liked_ids[start_idx:start_idx+chunk_size]
    query_idx = [int(book_to_idx[x]) for x in chunk]
    distances,neighbors = item_knn.kneighbors(R.T[query_idx],n_neighbors=MAX_ITEM_K+1)

    for j,bid in enumerate(chunk):
        ITEM_NEIGHBOR_CACHE[bid] = (neighbors[j][1:],np.maximum(1-distances[j][1:],0))

    done = min(start_idx+chunk_size,len(liked_ids))
    print(f'Item-KNN: {done}/{len(liked_ids)} ({100*done/len(liked_ids):.1f}%)',end='\r')

print(f'\nItem-KNN готов за {time.time()-start:.1f} сек.')

Кэширование User-KNN...
User-KNN готов: 200 пользователей за 0.2 сек.

Уникальных книг для Item-KNN: 4,021
Кэширование Item-KNN...
Item-KNN: 4021/4021 (100.0%)
Item-KNN готов за 4.2 сек.


In [14]:
def user_knn_scores(uid,candidate_ids):
    neighbors,sims = USER_NEIGHBOR_CACHE[int(uid)]
    idxs = [book_to_idx[x] for x in candidate_ids]
    values = np.asarray(R[neighbors][:,idxs].toarray())
    denom = sims.sum()+1e-8
    return dict(zip(candidate_ids,(values*sims[:,None]).sum(axis=0)/denom))

def item_knn_scores(uid,candidate_ids):
    candidate_set = set(candidate_ids)
    result = {int(x):0.0 for x in candidate_ids}
    liked = train_df[(train_df['user_id']==uid)&(train_df['rating']>=4)].sort_values('rating',ascending=False).head(PARAMS['CB_PROFILE_ITEMS'])

    for bid,rating in zip(liked['book_id'],liked['rating']):
        bid = int(bid)
        if bid not in ITEM_NEIGHBOR_CACHE: continue
        neighbors,sims = ITEM_NEIGHBOR_CACHE[bid]
        for nidx,sim in zip(neighbors,sims):
            book = int(book_ids[nidx])
            if book in candidate_set:
                result[book] += sim*float(rating)

    return result

def rank_knn_candidates(uid,candidate_ids,top_n=100):
    us = user_knn_scores(uid,candidate_ids)
    ins = item_knn_scores(uid,candidate_ids)
    rows = [(0.5*us.get(b,0)+0.5*ins.get(b,0),b) for b in candidate_ids]
    rows.sort(reverse=True)
    return [b for _,b in rows[:top_n]]

## Этап 3. Content-Based

Content-Based используется как финальный этап первой схемы гибрида и как этап генерации кандидатов во второй схеме.

Для книги формируется текст из `title`, `authors`, `genres` и `description`.

Используется TF-IDF с лучшей конфигурацией из предыдущей работы:

* `MIN_RATING_LIKE = 3`;
* `MAX_PROFILE_ITEMS = 50`;
* `NGRAM_RANGE = (1,5)`;
* `MAX_FEATURES = 30000`.

Профиль пользователя строится по понравившимся книгам, после чего вычисляется cosine similarity.


In [7]:
book_data = books_df.set_index('book_id').reindex(book_ids).fillna('')
book_texts = book_data['title']+' '+book_data['authors']+' '+book_data['genres'].str.replace('|',' ',regex=False)+' '+book_data['description']

tfidf_start = time.time()
vectorizer = TfidfVectorizer(max_features=PARAMS['CB_MAX_FEATURES'],ngram_range=PARAMS['CB_NGRAM_RANGE'],
                              min_df=2,max_df=0.95,sublinear_tf=True,strip_accents='unicode')
TFIDF = vectorizer.fit_transform(book_texts)
print(f'TF-IDF построен за {time.time()-tfidf_start:.1f} сек.; признаков: {TFIDF.shape[1]:,}')

def build_profile(uid):
    liked = train_df[(train_df['user_id']==uid)&(train_df['rating']>=PARAMS['CB_MIN_LIKE'])].sort_values('rating',ascending=False).head(PARAMS['CB_PROFILE_ITEMS'])
    if liked.empty: return None
    idxs = liked['book_id'].map(book_to_idx).to_numpy()
    weights = liked['rating'].to_numpy(dtype=np.float32)
    return normalize(csr_matrix(TFIDF[idxs].multiply(weights[:,None]).sum(axis=0)))

def cb_scores(uid,candidate_ids):
    profile = build_profile(uid)
    if profile is None: return {}
    idxs = [book_to_idx[x] for x in candidate_ids]
    scores = linear_kernel(profile,TFIDF[idxs]).ravel()
    return dict(zip(candidate_ids,scores))

TF-IDF построен за 18.9 сек.; признаков: 30,000


## Вариант A. SVD → User+Item KNN → Content-Based

Этот вариант соответствует последовательной каскадной схеме.

Сначала SVD формирует большой набор кандидатов.

Затем User+Item KNN уточняет этот набор и оставляет ограниченное количество наиболее поддержанных книг.

На последнем этапе Content-Based ранжирует только оставшийся набор по сходству с профилем пользователя.

Таким образом, Content-Based не работает со всем каталогом и не складывает свои оценки с KNN. Он является отдельным финальным этапом персонализации.


In [8]:
def hybrid_a(uid,svd_candidates=500,knn_pool=100,top_n=20):
    candidates = get_svd_candidates(uid,svd_candidates)
    candidates = rank_knn_candidates(uid,candidates,knn_pool)
    scores = cb_scores(uid,candidates)
    ranked = sorted(candidates,key=lambda x:scores.get(x,-1),reverse=True)[:top_n]
    return pd.DataFrame({'book_id':ranked,'score':[scores.get(x,0) for x in ranked]})

## Вариант B. Content-Based → User+Item KNN

Вторая схема использует обратный порядок.

Сначала Content-Based формирует содержательно близкий набор кандидатов.

После этого User+Item KNN оценивает только эти книги с учётом коллективных предпочтений пользователей.

Здесь Content-Based решает задачу ограничения пространства поиска и содержательной персонализации, а KNN выполняет окончательное ранжирование.

Это также является последовательной схемой, а не суммированием результатов двух моделей.


In [9]:
def hybrid_b(uid,cb_candidates=500,top_n=20):
    all_ids = [x for x in book_ids if x not in seen_cache[int(uid)]]
    cb = cb_scores(uid,all_ids)
    candidates = sorted(all_ids,key=lambda x:cb.get(x,-1),reverse=True)[:cb_candidates]
    ranked = rank_knn_candidates(uid,candidates,top_n)
    us = user_knn_scores(uid,ranked)
    ins = item_knn_scores(uid,ranked)
    rows = sorted([(0.5*us.get(b,0)+0.5*ins.get(b,0),b) for b in ranked],reverse=True)
    final = [b for _,b in rows[:top_n]]
    return pd.DataFrame({'book_id':final,'score':[s for s,_ in rows[:top_n]]})

## Расчёт метрик

Для каждой схемы рассчитываются Precision@5/10/20, Recall@5/10/20, NDCG@5/10/20 и HitRate@5/10/20.

Во время расчёта отображается прогресс обработки пользователей и примерное оставшееся время.

Основным критерием выбора конфигурации является NDCG@10. Дополнительно проверяются Precision@10, Recall@10 и HitRate@10.


In [10]:
def calc_metrics(rec_ids,relevant,k):
    rec_ids = rec_ids[:k]
    hits = [1 if x in relevant else 0 for x in rec_ids]
    h = sum(hits)
    precision = h/k
    recall = h/len(relevant) if relevant else 0
    hit_rate = 1.0 if h else 0.0
    dcg = sum(v/math.log2(i+2) for i,v in enumerate(hits))
    idcg = sum(1/math.log2(i+2) for i in range(min(k,len(relevant))))
    ndcg = dcg/idcg if idcg else 0
    return precision,recall,hit_rate,ndcg

def evaluate_model(func,label=''):
    store = {k:[] for k in PARAMS['TOP_K_VALUES']}
    start = time.time()
    for i,uid in enumerate(sample_eval_users,1):
        recs = func(int(uid),max(PARAMS['TOP_K_VALUES']))
        ids = recs['book_id'].tolist()
        relevant = test_relevant.get(uid,set())
        for k in PARAMS['TOP_K_VALUES']: store[k].append(calc_metrics(ids,relevant,k))
        if i==1 or i%10==0 or i==len(sample_eval_users):
            elapsed=time.time()-start
            speed=i/elapsed if elapsed else 0
            eta=(len(sample_eval_users)-i)/speed if speed else 0
            print(f'{label}: {i}/{len(sample_eval_users)} ({100*i/len(sample_eval_users):.1f}%), осталось {eta:.1f} сек.',end='\r')
    print()
    result={}
    for k,vals in store.items():
        m=np.mean(vals,axis=0)
        result.update({f'Precision@{k}':m[0],f'Recall@{k}':m[1],f'HitRate@{k}':m[2],f'NDCG@{k}':m[3]})
    return result

## Исследование гиперпараметров

Для первого варианта исследуются:

* число SVD-кандидатов;
* размер промежуточного KNN-пула.

Для второго варианта исследуется размер множества Content-Based кандидатов.

Параметры KNN и Content-Based, выбранные в предыдущих работах, используются как базовые.

Увеличение кандидатного пула повышает вероятность сохранить релевантную книгу, но увеличивает вычислительную стоимость последующего этапа. Слишком маленький пул может удалить релевантные объекты ещё до финального ранжирования.


In [23]:
hybrid_results = []
experiments = [('A', svd_n, knn_pool) for svd_n in PARAMS['SVD_CANDIDATES'] for knn_pool in PARAMS['KNN_POOL']]
experiments += [('B', cb_n, None) for cb_n in PARAMS['CB_CANDIDATES']]
total_exp = len(experiments)
all_start = time.time()

for exp_idx, (variant, p1, p2) in enumerate(experiments, 1):
    exp_start = time.time()
    print(f'\n{"="*70}')
    if variant == 'A':
        print(f'Эксперимент {exp_idx}/{total_exp}: SVD → KNN → CB | SVD candidates={p1}, KNN pool={p2}')
        metrics = evaluate_model(lambda uid,k: hybrid_a(uid,p1,p2,k), f'[{exp_idx}/{total_exp}]')
        metrics.update({'Model':'SVD → KNN → CB','SVD_CANDIDATES':p1,'KNN_POOL':p2,'CB_CANDIDATES':np.nan})
    else:
        print(f'Эксперимент {exp_idx}/{total_exp}: CB → KNN | CB candidates={p1}')
        metrics = evaluate_model(lambda uid,k: hybrid_b(uid,p1,k), f'[{exp_idx}/{total_exp}]')
        metrics.update({'Model':'CB → KNN','SVD_CANDIDATES':np.nan,'KNN_POOL':np.nan,'CB_CANDIDATES':p1})

    hybrid_results.append(metrics)

    elapsed = time.time() - all_start
    avg_time = elapsed / exp_idx
    eta = avg_time * (total_exp - exp_idx)
    print(f'Эксперимент завершён за {time.time()-exp_start:.1f} сек. | '
          f'Общий прогресс: {exp_idx}/{total_exp} ({100*exp_idx/total_exp:.1f}%) | '
          f'Осталось примерно: {eta:.1f} сек.')

hybrid_df = pd.DataFrame(hybrid_results)
print(f'\nВсе эксперименты завершены за {time.time()-all_start:.1f} сек.')

display(hybrid_df[['Model','SVD_CANDIDATES','KNN_POOL','CB_CANDIDATES',
                   'Precision@10','Recall@10','NDCG@10','HitRate@10']].round(6))


Эксперимент 1/14: SVD → KNN → CB | SVD candidates=300, KNN pool=50
[1/14]: 200/200 (100.0%), осталось 0.0 сек.
Эксперимент завершён за 5.1 сек. | Общий прогресс: 1/14 (7.1%) | Осталось примерно: 66.0 сек.

Эксперимент 2/14: SVD → KNN → CB | SVD candidates=300, KNN pool=100
[2/14]: 200/200 (100.0%), осталось 0.0 сек.
Эксперимент завершён за 5.0 сек. | Общий прогресс: 2/14 (14.3%) | Осталось примерно: 60.7 сек.

Эксперимент 3/14: SVD → KNN → CB | SVD candidates=300, KNN pool=200
[3/14]: 200/200 (100.0%), осталось 0.0 сек.
Эксперимент завершён за 5.1 сек. | Общий прогресс: 3/14 (21.4%) | Осталось примерно: 55.7 сек.

Эксперимент 4/14: SVD → KNN → CB | SVD candidates=500, KNN pool=50
[4/14]: 200/200 (100.0%), осталось 0.0 сек.
Эксперимент завершён за 5.1 сек. | Общий прогресс: 4/14 (28.6%) | Осталось примерно: 50.6 сек.

Эксперимент 5/14: SVD → KNN → CB | SVD candidates=500, KNN pool=100
[5/14]: 200/200 (100.0%), осталось 0.0 сек.
Эксперимент завершён за 5.3 сек. | Общий прогресс: 5/14 (3

,Model,SVD_CANDIDATES,KNN_POOL,CB_CANDIDATES,Precision@10,Recall@10,NDCG@10,HitRate@10
0,SVD → KNN → CB,300.0,50.0,NaN,0.1900,0.134517,0.224067,0.770
1,SVD → KNN → CB,300.0,100.0,NaN,0.1625,0.113784,0.195572,0.720
2,SVD → KNN → CB,300.0,200.0,NaN,0.1445,0.102789,0.176936,0.660
3,SVD → KNN → CB,500.0,50.0,NaN,0.1960,0.138368,0.232253,0.780
4,SVD → KNN → CB,500.0,100.0,NaN,0.1730,0.119940,0.205625,0.740
5,SVD → KNN → CB,500.0,200.0,NaN,0.1520,0.105936,0.184163,0.660
6,SVD → KNN → CB,800.0,50.0,NaN,0.2035,0.143202,0.238144,0.795
7,SVD → KNN → CB,800.0,100.0,NaN,0.1760,0.120848,0.208350,0.745
8,SVD → KNN → CB,800.0,200.0,NaN,0.1570,0.107913,0.186913,0.685
9,CB → KNN,NaN,NaN,200.0,0.2380,0.162680,0.290740,0.850


## Выбор лучшей конфигурации

Основным критерием является NDCG@10.

После исследования выбирается конфигурация с максимальным NDCG@10. Дополнительно проверяются Precision@10, Recall@10 и HitRate@10.

Особое внимание уделяется сравнению с предыдущей User+Item KNN системой. Для выполнения условия задания желательно, чтобы итоговый гибрид превосходил её по всем четырём основным метрикам.

Если несколько конфигураций имеют близкие значения NDCG@10, предпочтение отдаётся более простой и быстрой конфигурации с меньшим числом кандидатов.


In [24]:
best_idx = hybrid_df['NDCG@10'].idxmax()
best_hybrid = hybrid_df.loc[best_idx]

print('Лучшая конфигурация гибрида:')
display(best_hybrid.to_frame().round(6))

baseline_knn = {'Precision@10':0.210606,'Recall@10':0.144350,'NDCG@10':0.259965,'HitRate@10':0.833333}
print('\nСравнение с User+Item KNN:')
for metric in baseline_knn:
    delta = best_hybrid[metric]-baseline_knn[metric]
    print(f'{metric}: {delta:+.6f}')

condition = all(best_hybrid[m] > baseline_knn[m] for m in baseline_knn)
print(f'\nВсе четыре метрики выше KNN: {condition}')

Лучшая конфигурация гибрида:


,11
Precision@5,0.314
Recall@5,0.116051
HitRate@5,0.755
NDCG@5,0.342841
Precision@10,0.248
Recall@10,0.173076
HitRate@10,0.86
NDCG@10,0.296498
Precision@20,0.183
Recall@20,0.248413



Сравнение с User+Item KNN:
Precision@10: +0.037394
Recall@10: +0.028726
NDCG@10: +0.036533
HitRate@10: +0.026667

Все четыре метрики выше KNN: True


## Сравнение с предыдущими моделями

Для итогового сравнения используются лучшие или основные конфигурации предыдущих работ:

* User+Item KNN;
* SVD(10);
* Content-Based;
* лучший найденный гибрид.

SVD(10) используется как основной вариант SVD, поскольку он показал лучший результат по NDCG@10 среди исследованных вариантов SVD.

Content-Based используется с лучшей найденной конфигурацией из предыдущей работы.


In [25]:
previous_df = pd.DataFrame({
    'Precision@10':[0.210606,0.106000,0.118000,best_hybrid['Precision@10']],
    'Recall@10':[0.144350,0.073089,0.081363,best_hybrid['Recall@10']],
    'NDCG@10':[0.259965,0.140460,0.137445,best_hybrid['NDCG@10']],
    'HitRate@10':[0.833333,0.535000,0.580000,best_hybrid['HitRate@10']]
},index=['User+Item KNN','SVD (10)','Content-Based','Hybrid (best)'])

display(previous_df.round(6))

,Precision@10,Recall@10,NDCG@10,HitRate@10
User+Item KNN,0.210606,0.144350,0.259965,0.833333
SVD (10),0.106000,0.073089,0.140460,0.535000
Content-Based,0.118000,0.081363,0.137445,0.580000
Hybrid (best),0.248000,0.173076,0.296498,0.860000


## Демонстрация на конкретном пользователе

Для одного пользователя показываются:

1. книги, которым пользователь поставил высокие оценки;
2. кандидаты после первого этапа;
3. итоговые рекомендации гибридной системы.

Это позволяет проверить, как последовательное применение моделей изменяет набор рекомендаций.


In [26]:
example_user = int(sample_eval_users[0])

print(f'Пользователь: {example_user}')
print('\nКниги с высокими оценками:')
display(train_df[train_df['user_id']==example_user].sort_values('rating',ascending=False).head(10))

if best_hybrid['Model']=='SVD → KNN → CB':
    example_recs = hybrid_a(example_user,int(best_hybrid['SVD_CANDIDATES']),int(best_hybrid['KNN_POOL']),10)
else:
    example_recs = hybrid_b(example_user,int(best_hybrid['CB_CANDIDATES']),10)

example_display = example_recs.merge(books_df[['book_id','title','authors','genres']],on='book_id',how='left')

print('\nИтоговые рекомендации:')
display(example_display)

Пользователь: 34618

Книги с высокими оценками:


,user_id,book_id,rating
5756153,34618,6928,5.0
5365879,34618,772,5.0
5102987,34618,4332,5.0
3960749,34618,4684,5.0
5614228,34618,4503,5.0
5516559,34618,130,4.0
5518884,34618,58,4.0
3968499,34618,4330,4.0
5516561,34618,2709,4.0
4095770,34618,28,4.0



Итоговые рекомендации:


,book_id,score,title,authors,genres
0,8,10.902780,The Catcher in the Rye,['J.D. Salinger'],"['classics', 'fiction', 'young-adult']"
1,29,10.590504,Romeo and Juliet,"['William Shakespeare', 'Robert Jack...","['classics', 'fiction', 'romance', 'poetry']"
2,4,10.296572,To Kill a Mockingbird,['Harper Lee'],"['classics', 'fiction', 'historical-fiction', ..."
3,32,10.178352,Of Mice and Men,['John Steinbeck'],"['classics', 'fiction', 'historical-fiction']"
4,131,8.925550,The Grapes of Wrath,['John Steinbeck'],"['classics', 'fiction', 'historical-fiction']"
5,65,8.016175,Slaughterhouse-Five,['Kurt Vonnegut Jr.'],"['classics', 'fiction', 'science-fiction', 'hi..."
6,116,7.843954,The Adventures of Tom Sawyer,"['Mark Twain', 'Guy Cardwell', 'John Seelye']","['classics', 'fiction', 'young-adult', 'histor..."
7,113,7.063584,Catch-22,['Joseph Heller'],"['classics', 'fiction', 'historical-fiction']"
8,301,6.669873,Heart of Darkness,['Joseph Conrad'],"['classics', 'fiction', 'historical-fiction']"
9,341,5.878599,The Iliad,"['Homer', 'Robert Fagles', 'Frédéric Mugler', ...","['classics', 'poetry', 'fiction', 'fantasy', '..."


## Интерпретация результатов

Лучшей конфигурацией стала схема **Content-Based → KNN** с `CB_CANDIDATES = 800`. Она показала лучшие значения всех основных метрик среди исследованных гибридных вариантов.

Вариант **SVD → KNN → Content-Based** также дал хорошие результаты, однако во всех исследованных конфигурациях уступил исходному User+Item KNN. При увеличении `SVD_CANDIDATES` качество при `KNN_POOL = 50` росло, а увеличение `KNN_POOL` приводило к снижению качества. Это показывает, что слишком большой промежуточный набор кандидатов ухудшает финальную персонализацию Content-Based.

Для схемы **Content-Based → KNN** увеличение числа кандидатов с 200 до 800 постепенно повышало качество. Это показывает, что Content-Based хорошо работает как этап предварительного отбора, если перед KNN сохраняется достаточно широкий набор содержательно подходящих книг.

Лучшая гибридная модель превзошла User+Item KNN по Precision@10, Recall@10, NDCG@10 и HitRate@10. По сравнению с отдельным Content-Based также наблюдается улучшение всех основных метрик. Это подтверждает эффективность последовательного объединения Content-Based и KNN.

Таким образом, в данной работе наиболее удачным оказался порядок **Content-Based → User+Item KNN**: сначала отбираются книги, соответствующие содержательному профилю пользователя, после чего KNN уточняет итоговый порядок с учётом коллективных предпочтений.


## Выводы

В работе была разработана итоговая гибридная рекомендательная система с обязательным использованием Content-Based.

Были исследованы две последовательные схемы: `SVD → KNN → Content-Based` и `Content-Based → KNN`. Такой подход не использует простое сложение оценок моделей, а последовательно решает задачи поиска и уточнения кандидатов.

Лучшей стала схема **Content-Based → KNN** с `CB_CANDIDATES = 800`. Она превзошла предыдущие User+Item KNN, SVD(10) и Content-Based по всем четырём рассматриваемым метрикам.

Эксперименты также показали, что размер кандидатного множества существенно влияет на результат. Для схемы Content-Based → KNN увеличение числа кандидатов улучшало качество, а для SVD → KNN → Content-Based слишком большой промежуточный KNN-пул, наоборот, снижал его.

Итоговый результат показывает, что последовательное объединение содержательной информации о книгах и пользовательских взаимодействий позволяет получить более качественные рекомендации, чем каждый из рассмотренных подходов по отдельности.
